# 🔢 Notebook 05 — Embeddings & FAISS Vector Store
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the labelled dataset
- Build formatted text chunks (question + context + answer)
- Generate dense embeddings with `all-MiniLM-L6-v2`
- Build a FAISS `IndexFlatL2` index
- Persist index + chunk mapping to disk
- Sanity-check retrieval with 5 test queries (top-3)
- Measure retrieval latency (KPI: < 500ms)

### 📋 Deliverables
- `data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss`
- `data/embeddings/faiss_index/chunk_mapping.csv`
- `data/embeddings/faiss_index/chunk_mapping.pkl`

---

## 1. Imports & Setup

In [1]:
import os
import pickle
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("✅ Imports successful")

D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Imports successful


## 2. Load Data & Build Text Chunks

Each chunk combines **question + context + answer** into a single
retrieval unit. Including the context (medical abstract) is critical
because it contains the evidence a RAG pipeline needs to ground its answers.

In [2]:
data_path = "../data/processed/pubmedqa_labelled.csv"

df = pd.read_csv(data_path)
print(f"✅ Loaded: {data_path}")
print(f"   Total rows: {len(df):,}")
print(f"   Columns: {list(df.columns)}")

# Validate required columns
required_columns = {"question", "context", "answer"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Drop rows with missing text
before = len(df)
df = df.dropna(subset=["question", "context", "answer"]).copy()
df["question"] = df["question"].astype(str).str.strip()
df["context"] = df["context"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df = df[(df["question"] != "") & (df["context"] != "") & (df["answer"] != "")]
df = df.reset_index(drop=True)
print(f"   Rows after cleaning: {len(df):,} (dropped {before - len(df)})")

✅ Loaded: ../data/processed/pubmedqa_labelled.csv
   Total rows: 10,000
   Columns: ['question', 'context', 'answer', 'category']
   Rows after cleaning: 10,000 (dropped 0)


In [3]:
# Build text chunks: question + context + answer
# This gives the retriever maximum signal for semantic matching
df["text_chunk"] = (
    "Question: " + df["question"]
    + "\nContext: " + df["context"]
    + "\nAnswer: " + df["answer"]
)

text_chunks = df["text_chunk"].tolist()
n_chunks = len(text_chunks)

print(f"\n✅ Prepared {n_chunks:,} text chunks")
print(f"\nSample chunk (first 500 chars):\n")
print(text_chunks[0][:500])


✅ Prepared 10,000 text chunks

Sample chunk (first 500 chars):

Question: Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?
Context: Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to women receiving conventional therapy in the clinical setting. To compare naturopathic therapy with conventional medical therapy for treatment of selected menopausal symptoms. A retrospective coho


## 3. Generate Embeddings

Using `sentence-transformers/all-MiniLM-L6-v2`:
- 384-dimensional embeddings
- Fast inference, good semantic quality
- FAISS requires `float32` arrays

In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
print(f"✅ Loaded model: {model_name}")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

# %%
print(f"⏳ Encoding {n_chunks:,} chunks...")

start_time = time.time()

embeddings = model.encode(
    text_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

encoding_time = time.time() - start_time

# FAISS requires float32
embeddings = np.asarray(embeddings, dtype=np.float32)

print(f"\n✅ Encoding complete!")
print(f"   Shape: {embeddings.shape}")
print(f"   Dtype: {embeddings.dtype}")
print(f"   Time: {encoding_time:.1f}s ({encoding_time/n_chunks*1000:.1f}ms per chunk)")

D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Loaded model: sentence-transformers/all-MiniLM-L6-v2
   Embedding dimension: 384
⏳ Encoding 10,000 chunks...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


✅ Encoding complete!
   Shape: (10000, 384)
   Dtype: float32
   Time: 14.0s (1.4ms per chunk)


## 4. Build FAISS Index

`IndexFlatL2` — exact nearest-neighbor search using L2 distance.
- No training needed
- Dimension inferred from embeddings
- Lower distance = higher similarity

In [5]:
d = embeddings.shape[1]

index = faiss.IndexFlatL2(d)
index.add(embeddings)

print(f"✅ FAISS index built")
print(f"   Dimension: {d}")
print(f"   Total vectors: {index.ntotal:,}")

✅ FAISS index built
   Dimension: 384
   Total vectors: 10,000


## 5. Save Index & Chunk Mapping

In [6]:
output_dir = Path("../data/embeddings/faiss_index")
output_dir.mkdir(parents=True, exist_ok=True)

index_path = output_dir / "pubmedqa_index_flatl2.faiss"
mapping_csv_path = output_dir / "chunk_mapping.csv"
mapping_pkl_path = output_dir / "chunk_mapping.pkl"

# Save FAISS index
faiss.write_index(index, str(index_path))

# Save mapping table (chunk_id → question, context, answer, category, text_chunk)
mapping_df = df[["question", "context", "answer", "category", "text_chunk"]].copy()
mapping_df.insert(0, "chunk_id", np.arange(len(mapping_df), dtype=np.int32))

mapping_df.to_csv(mapping_csv_path, index=False)

with open(mapping_pkl_path, "wb") as f:
    pickle.dump(mapping_df, f)

print(f"✅ Saved FAISS index to: {index_path}")
print(f"   File size: {os.path.getsize(index_path) / 1024**2:.1f} MB")
print(f"✅ Saved chunk mapping CSV to: {mapping_csv_path}")
print(f"✅ Saved chunk mapping PKL to: {mapping_pkl_path}")

✅ Saved FAISS index to: ..\data\embeddings\faiss_index\pubmedqa_index_flatl2.faiss
   File size: 14.6 MB
✅ Saved chunk mapping CSV to: ..\data\embeddings\faiss_index\chunk_mapping.csv
✅ Saved chunk mapping PKL to: ..\data\embeddings\faiss_index\chunk_mapping.pkl


In [7]:
# ── Auto-upload to HuggingFace ────────────────────────────────────────────
import sys
sys.path.append(os.path.abspath('..'))

from src.data.hub import upload_file

print("\n📤 Syncing to HuggingFace...")
upload_file(
    "data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss",
    "embeddings/pubmedqa_index_flatl2.faiss"
)
upload_file(
    "data/embeddings/faiss_index/chunk_mapping.pkl",
    "embeddings/chunk_mapping.pkl"
)
upload_file(
    "data/embeddings/faiss_index/chunk_mapping.csv",
    "embeddings/chunk_mapping.csv"
)


📤 Syncing to HuggingFace...


pubmedqa_index_flatl2.faiss:   0%|          | 0.00/15.4M [00:00<?, ?B/s]

  ✅ Uploaded: data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss (14.6 MB)


chunk_mapping.pkl:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

  ✅ Uploaded: data/embeddings/faiss_index/chunk_mapping.pkl (33.1 MB)


chunk_mapping.csv:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

  ✅ Uploaded: data/embeddings/faiss_index/chunk_mapping.csv (33.1 MB)


True

## 6. Sanity-Check Retrieval (Top-3) + Latency Measurement

**M2 KPI: FAISS returns results in < 500ms**

In [8]:
test_queries = [
    "What are effective treatments for irritable bowel syndrome symptoms?",
    "Can hypothyroidism increase risk of fatty liver disease?",
    "Is laparoscopic prostatectomy superior to retropubic surgery?",
    "How accurate is diagnosis of acute otitis media in primary care?",
    "Does secondary isoniazid therapy reduce recurrent tuberculosis in HIV patients?",
]

# Encode queries
query_embeddings = model.encode(
    test_queries,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

k = 3  # top-3 results

# ── Run retrieval with timing ────────────────────────────────────────────
latencies = []

for qi, query in enumerate(test_queries):
    # Time only the FAISS search (not encoding)
    start = time.time()
    D, I = index.search(query_embeddings[qi:qi+1], k)
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)

    print("=" * 110)
    print(f"Query {qi + 1}: {query}")
    print(f"⏱️  Retrieval latency: {elapsed_ms:.2f}ms")
    print()

    for rank in range(k):
        idx = int(I[0, rank])
        dist = float(D[0, rank])
        chunk = mapping_df.loc[idx, "text_chunk"]

        print(f"  Top {rank + 1} | Chunk ID: {idx} | L2 Distance: {dist:.4f}")
        print(f"  {chunk[:300]}")
        if len(chunk) > 300:
            print("  ... [truncated]")
        print("-" * 110)

# ── Latency summary ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("⏱️  LATENCY SUMMARY")
print("=" * 60)
print(f"  Min:    {min(latencies):.2f}ms")
print(f"  Max:    {max(latencies):.2f}ms")
print(f"  Mean:   {np.mean(latencies):.2f}ms")
print(f"  Median: {np.median(latencies):.2f}ms")
print()

if max(latencies) < 500:
    print("✅ KPI MET: All queries returned in < 500ms")
else:
    print("⚠️  KPI NOT MET: Some queries exceeded 500ms")

Query 1: What are effective treatments for irritable bowel syndrome symptoms?
⏱️  Retrieval latency: 6.00ms

  Top 1 | Chunk ID: 32 | L2 Distance: 0.9456
  Question: Does depression influence symptom severity in irritable bowel syndrome?
Context: Irritable bowel syndrome (IBS) is frequently associated with mood disorder. However, it is typically difficult to distinguish between disturbed mood as a causal agent and disturbed mood as a consequence of the
  ... [truncated]
--------------------------------------------------------------------------------------------------------------
  Top 2 | Chunk ID: 7299 | L2 Distance: 0.9781
  Question: Is visceral hypersensitivity correlated with symptom severity in children with functional gastrointestinal disorders?
Context: Abdominal pain related to irritable bowel syndrome (IBS) and functional abdominal pain (FAP) is frequent in children and can be of variable severity. Both IBS and 
  ... [truncated]
----------------------------------------------

## ✅ Summary

| Item | Result |
|---|---|
| Text chunks | Question + Context + Answer |
| Embedding model | `all-MiniLM-L6-v2` (384d) |
| FAISS index type | `IndexFlatL2` |
| Vectors stored | {n_chunks:,} |
| Sanity check | 5 queries × top-3 results |
| Latency KPI | < 500ms per query |

### Files Saved
- `data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss`
- `data/embeddings/faiss_index/chunk_mapping.csv`
- `data/embeddings/faiss_index/chunk_mapping.pkl`

---

### ➡️ Next Step
Open **`05_rag_pipeline.ipynb`** to build the LangChain RAG pipeline.